In [ ]:
# ==========================================
# 0. КЛОНИРОВАНИЕ РЕПОЗИТОРИЯ И УСТАНОВКА
# ==========================================
import os

REPO_URL = "https://github.com/bald-monkey-perapinak/box-solution-rubezhka-kazantseva.git"
REPO_DIR = "box-solution-rubezhka-kazantseva"

if not os.path.exists(REPO_DIR):
    !git clone {REPO_URL}
    print("✅ Репозиторий клонирован!")
else:
    !git -C {REPO_DIR} pull
    print("✅ Репозиторий обновлён!")

os.chdir(REPO_DIR)
print(f"📁 Рабочая директория: {os.getcwd()}")
print("Файлы в репозитории:")
!ls -la

# Установка зависимостей из requirements.txt
if os.path.exists("requirements.txt"):
    !pip install -r requirements.txt -q
    print("✅ Зависимости установлены!")


## 🥊 Шаг 1: Извлечение ключевых точек (YOLO Pose)

In [ ]:
# ==========================================
# МАГИЯ ОТ ЗАВИСАНИЙ: Глушим внутренние потоки библиотек
# (Писать строго до импорта cv2 и torch!)
# ==========================================
import os
os.environ["OMP_NUM_THREADS"] = "1"
os.environ["OPENBLAS_NUM_THREADS"] = "1"
os.environ["MKL_NUM_THREADS"] = "1"
import cv2
cv2.setNumThreads(0)

import numpy as np
import pandas as pd
import json
from ultralytics import YOLO
from scipy.optimize import linear_sum_assignment
import multiprocessing

# ==========================================
# 1. КОНСТАНТЫ И НАСТРОЙКИ
# ==========================================
L_SHOULDER, R_SHOULDER = 5, 6
L_ELBOW, R_ELBOW = 7, 8
L_WRIST, R_WRIST = 9, 10
L_HIP, R_HIP = 11, 12
L_KNEE, R_KNEE = 13, 14

SKELETON_PAIRS = [
    (5, 6), (5, 7), (7, 9), (6, 8), (8, 10),
    (5, 11), (6, 12), (11, 12), (11, 13), (13, 15), (12, 14), (14, 16)
]

LOWER_RED1, UPPER_RED1 = np.array([0, 100, 70]), np.array([12, 255, 255])
LOWER_RED2, UPPER_RED2 = np.array([165, 100, 70]), np.array([180, 255, 255])
LOWER_BLUE, UPPER_BLUE = np.array([90, 100, 70]), np.array([130, 255, 255])
LOWER_WHITE, UPPER_WHITE = np.array([0, 0, 150]), np.array([180, 60, 255])

# ==========================================
# 2. ФУНКЦИИ АНАЛИЗА ЦВЕТА И РОЛЕЙ
# ==========================================
def get_polygon_crop_and_mask(frame, target_kps):
    frame_h, frame_w = frame.shape[:2]
    if any(kp[2] < 0.4 for kp in target_kps): return None, None
    pts = np.array([[kp[0], kp[1]] for kp in target_kps], dtype=np.int32)
    x, y, w, h = cv2.boundingRect(pts)
    x1, y1 = max(0, x), max(0, y)
    x2, y2 = min(frame_w, x + w), min(frame_h, y + h)
    if x2 <= x1 or y2 <= y1: return None, None
    crop = frame[y1:y2, x1:x2]
    mask = np.zeros((crop.shape[0], crop.shape[1]), dtype=np.uint8)
    pts_shifted = pts - [x1, y1]
    cv2.fillPoly(mask, [pts_shifted], 255)
    return crop, mask

def get_glove_crop_and_mask(frame, elbow_kp, wrist_kp):
    if elbow_kp[2] < 0.4 or wrist_kp[2] < 0.4: return None, None
    ex, ey = elbow_kp[:2]
    wx, wy = wrist_kp[:2]
    vx, vy = wx - ex, wy - ey
    gx, gy = int(wx + vx * 0.4), int(wy + vy * 0.4)
    frame_h, frame_w = frame.shape[:2]
    x1, y1 = max(0, gx - 15), max(0, gy - 15)
    x2, y2 = min(frame_w, gx + 15), min(frame_h, gy + 15)
    if x2 <= x1 or y2 <= y1: return None, None
    crop = frame[y1:y2, x1:x2]
    mask = np.zeros((crop.shape[0], crop.shape[1]), dtype=np.uint8)
    cx, cy = gx - x1, gy - y1
    cv2.circle(mask, (cx, cy), 15, 255, -1)
    return crop, mask

def extract_colors(crop, mask):
    hsv = cv2.cvtColor(crop, cv2.COLOR_BGR2HSV)
    roi_pixels = cv2.bitwise_and(hsv, hsv, mask=mask)
    total_pixels = np.count_nonzero(mask)
    if total_pixels == 0: return 0.0, 0.0, 0.0
    mask_red = cv2.bitwise_or(cv2.inRange(roi_pixels, LOWER_RED1, UPPER_RED1),
                              cv2.inRange(roi_pixels, LOWER_RED2, UPPER_RED2))
    mask_blue = cv2.inRange(roi_pixels, LOWER_BLUE, UPPER_BLUE)
    mask_white = cv2.inRange(roi_pixels, LOWER_WHITE, UPPER_WHITE)
    return (np.sum(mask_red > 0) / total_pixels,
            np.sum(mask_blue > 0) / total_pixels,
            np.sum(mask_white > 0) / total_pixels)

def classify_person_features(frame, kps):
    total_red, total_blue, total_ref = 0.0, 0.0, 0.0
    weight_sum = 0.0001
    torso_kps = [kps[L_SHOULDER], kps[R_SHOULDER], kps[R_HIP], kps[L_HIP]]
    torso_crop, torso_mask = get_polygon_crop_and_mask(frame, torso_kps)
    if torso_crop is not None:
        conf_w = (sum(kp[2] for kp in torso_kps) / 4) ** 3
        r, b, w = extract_colors(torso_crop, torso_mask)
        total_red += r * conf_w; total_blue += b * conf_w; total_ref += w * conf_w
        weight_sum += conf_w
    shorts_kps = [kps[L_HIP], kps[R_HIP], kps[R_KNEE], kps[L_KNEE]]
    shorts_crop, shorts_mask = get_polygon_crop_and_mask(frame, shorts_kps)
    if shorts_crop is not None:
        conf_w = (sum(kp[2] for kp in shorts_kps) / 4) ** 3
        r, b, _ = extract_colors(shorts_crop, shorts_mask)
        total_red += r * conf_w; total_blue += b * conf_w
        weight_sum += conf_w
    for elbow_idx, wrist_idx in [(L_ELBOW, L_WRIST), (R_ELBOW, R_WRIST)]:
        g_crop, g_mask = get_glove_crop_and_mask(frame, kps[elbow_idx], kps[wrist_idx])
        if g_crop is not None:
            conf_w = ((kps[elbow_idx][2] + kps[wrist_idx][2]) / 2) ** 3 * 0.7
            r, b, _ = extract_colors(g_crop, g_mask)
            total_red += r * conf_w; total_blue += b * conf_w
            weight_sum += conf_w
    return total_red / weight_sum, total_blue / weight_sum, total_ref / weight_sum

class BoxingRoleManager:
    def __init__(self):
        self.prev_centers = {'RED': None, 'BLUE': None}

    def update(self, frame, track_ids, bboxes, keypoints):
        frame_h, frame_w = frame.shape[:2]
        frame_diag = np.sqrt(frame_w**2 + frame_h**2)
        candidates = []
        for tid, bbox, kps in zip(track_ids, bboxes, keypoints):
            if (bbox[3] - bbox[1]) < frame_h * 0.2: continue
            red, blue, ref = classify_person_features(frame, kps)
            if ref > 0.15 and red < 0.1 and blue < 0.1: continue
            cx, cy = (bbox[0] + bbox[2]) / 2, (bbox[1] + bbox[3]) / 2
            candidates.append({
                'tid': tid, 'bbox': bbox, 'kps': kps, 'center': (cx, cy),
                'score_red': red, 'score_blue': blue
            })
        output_fighters = {}
        if not candidates: return output_fighters
        cost_matrix = np.zeros((len(candidates), 2))
        for i, cand in enumerate(candidates):
            s_red, s_blue = cand['score_red'], cand['score_blue']
            if self.prev_centers['RED']:
                dist = np.linalg.norm(np.array(cand['center']) - np.array(self.prev_centers['RED']))
                mem_score = max(0.0, 1.0 - (dist / (frame_diag * 0.3)))
                s_red = s_red * 0.6 + mem_score * 0.4
            if self.prev_centers['BLUE']:
                dist = np.linalg.norm(np.array(cand['center']) - np.array(self.prev_centers['BLUE']))
                mem_score = max(0.0, 1.0 - (dist / (frame_diag * 0.3)))
                s_blue = s_blue * 0.6 + mem_score * 0.4
            cost_matrix[i, 0] = 1.0 - s_red
            cost_matrix[i, 1] = 1.0 - s_blue
        row_ind, col_ind = linear_sum_assignment(cost_matrix)
        for r, c in zip(row_ind, col_ind):
            role = 'RED' if c == 0 else 'BLUE'
            if cost_matrix[r, c] < 0.85:
                output_fighters[role] = candidates[r]
                self.prev_centers[role] = candidates[r]['center']
        if 'RED' not in output_fighters: self.prev_centers['RED'] = None
        if 'BLUE' not in output_fighters: self.prev_centers['BLUE'] = None
        return output_fighters

# ==========================================
# 3. ИЗВЛЕЧЕНИЕ ФИЧЕЙ С ПРОПУСКОМ КАДРОВ
# ==========================================
def process_video_task(args):
    input_path, output_csv_path, model_name = args
    model = YOLO(model_name)
    cap = cv2.VideoCapture(input_path)
    if not cap.isOpened():
        return f"❌ Ошибка открытия: {os.path.basename(input_path)}"
    total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    manager = BoxingRoleManager()
    frame_idx = 0
    extracted_data = []
    print(f"▶️ Старт: {os.path.basename(input_path)} | Всего кадров: {total_frames}", flush=True)
    while cap.isOpened():
        ret, frame = cap.read()
        if not ret:
            break
        if frame_idx % 10 != 0:
            frame_idx += 1
            continue
        results = model.track(frame, persist=True, classes=[0], conf=0.35, verbose=False)
        frame_data = {'frame': frame_idx, 'red_kps': None, 'blue_kps': None}
        if results[0].boxes is not None and results[0].boxes.id is not None:
            bboxes = results[0].boxes.xyxy.cpu().numpy()
            track_ids = results[0].boxes.id.int().cpu().numpy()
            keypoints = results[0].keypoints.data.cpu().numpy()
            fighters = manager.update(frame, track_ids, bboxes, keypoints)
            if 'RED' in fighters:
                frame_data['red_kps'] = json.dumps(fighters['RED']['kps'].tolist())
            if 'BLUE' in fighters:
                frame_data['blue_kps'] = json.dumps(fighters['BLUE']['kps'].tolist())
        extracted_data.append(frame_data)
        if frame_idx % 300 == 0:
            percent = (frame_idx / total_frames) * 100 if total_frames > 0 else 0
            print(f"[{os.path.basename(input_path)}] Прогресс: {frame_idx}/{total_frames} ({percent:.1f}%)", flush=True)
        frame_idx += 1
    cap.release()
    df = pd.DataFrame(extracted_data)
    df.to_csv(output_csv_path, index=False)
    print(f"✅ Готово: {os.path.basename(input_path)}", flush=True)
    return f"Успех: {os.path.basename(input_path)}"

# ==========================================
# 4. МУЛЬТИПРОЦЕССОРНЫЙ ЗАПУСК
# ==========================================
# ⚠️ Укажи путь к папке с видео (на Google Drive или Kaggle)
INPUT_DATA_ROOT = '/data'          # <-- ИЗМЕНИ если нужно
OUTPUT_FEATURES_DIR = './extracted_features'
MODEL_NAME = 'yolov8m-pose.pt'    # YOLO Pose модель (скачается автоматически)

os.makedirs(OUTPUT_FEATURES_DIR, exist_ok=True)

# Проверка: есть ли видео для обработки?
tasks = []
if os.path.isdir(INPUT_DATA_ROOT):
    for root, dirs, files in os.walk(INPUT_DATA_ROOT):
        for filename in files:
            if filename.endswith((".mp4", ".MOV", ".mov", ".avi")):
                input_video = os.path.join(root, filename)
                parent_folder = os.path.basename(root).replace(" ", "_")
                base_name = os.path.splitext(filename)[0].replace(" ", "_")
                csv_filename = f"{parent_folder}_{base_name}_features.csv"
                output_csv = os.path.join(OUTPUT_FEATURES_DIR, csv_filename)
                if not os.path.exists(output_csv):
                    tasks.append((input_video, output_csv, MODEL_NAME))

print(f"Всего видео для обработки: {len(tasks)}")

if tasks:
    print(f"Запускаем обработку видео...")
    _ = YOLO(MODEL_NAME)  # Предзагрузка модели
    for task in tasks:
        process_video_task(task)
else:
    if not os.path.isdir(INPUT_DATA_ROOT):
        print(f"⚠️  Папка {INPUT_DATA_ROOT} не найдена. Если файлы extracted_features/*.csv уже готовы — пропусти эту ячейку.")
    else:
        print("✅ Вся обработка уже завершена!")


## 📊 Шаг 2: Сборка датасета из CSV-файлов репозитория

In [ ]:
import pandas as pd
import glob
import os

# ──────────────────────────────────────────────────────────
# ВАРИАНТ A: если extracted_features/*.csv уже есть локально
# ВАРИАНТ B: если репо содержит готовый final_clean_features.csv
# ──────────────────────────────────────────────────────────

folder_path = 'extracted_features'

if os.path.isdir(folder_path) and len(glob.glob(os.path.join(folder_path, "*.csv"))) > 0:
    # ── ВАРИАНТ A: собираем из локально извлечённых файлов ──
    all_files = glob.glob(os.path.join(folder_path, "*.csv"))
    df_list = []
    for file in all_files:
        try:
            df = pd.read_csv(file)
            base_name = os.path.basename(file).replace('_features.csv', '.mp4')
            df['video_name'] = base_name
            df_list.append(df)
        except Exception as e:
            print(f"Ошибка чтения {file}: {e}")
    full_features_df = pd.concat(df_list, ignore_index=True)
    print(f"✅ Успешно склеено файлов: {len(df_list)}")
    print(f"📊 Всего строк (кадров) в датасете: {len(full_features_df)}")

elif os.path.exists('final_clean_features.csv'):
    # ── ВАРИАНТ B: файл уже готов в репозитории ──
    print("📥 Используем final_clean_features.csv из репозитория...")
    full_features_df = pd.read_csv('final_clean_features.csv')
    print(f"📊 Строк: {len(full_features_df)}, Колонок: {len(full_features_df.columns)}")

else:
    raise FileNotFoundError(
        "Не найдены ни папка extracted_features/, ни final_clean_features.csv. "
        "Запусти Шаг 1 или проверь репозиторий."
    )

display(full_features_df.head())


## 🔬 Шаг 3: Расчёт продвинутых боксёрских фичей

In [ ]:
import json
import numpy as np
import pandas as pd
import math

def get_angle(a, b, c):
    ba = a - b
    bc = c - b
    cosine_angle = np.dot(ba, bc) / (np.linalg.norm(ba) * np.linalg.norm(bc) + 1e-6)
    angle = np.arccos(np.clip(cosine_angle, -1.0, 1.0))
    return np.degrees(angle)

def calculate_advanced_features(row):
    if pd.isna(row.get('red_kps')) or pd.isna(row.get('blue_kps')):
        return pd.Series({
            'dist_heads': np.nan,
            'red_l_ext': np.nan, 'red_r_ext': np.nan,
            'blue_l_ext': np.nan, 'blue_r_ext': np.nan,
            'red_stance_width': np.nan, 'blue_stance_width': np.nan,
            'red_lean_x': np.nan, 'blue_lean_x': np.nan,
            'red_l_angle': np.nan, 'red_r_angle': np.nan,
            'blue_l_angle': np.nan, 'blue_r_angle': np.nan,
            'red_l_elbow_y_rel': np.nan, 'red_r_elbow_y_rel': np.nan,
            'blue_l_elbow_y_rel': np.nan, 'blue_r_elbow_y_rel': np.nan,
            'red_l_wrist_y': np.nan, 'red_r_wrist_y': np.nan,
            'blue_l_wrist_y': np.nan, 'blue_r_wrist_y': np.nan
        })
    try:
        red = np.array(json.loads(row['red_kps']))
        blue = np.array(json.loads(row['blue_kps']))
        red_head, blue_head = red[0][:2], blue[0][:2]
        dist_heads = np.linalg.norm(red_head - blue_head)
        red_l_ext = np.linalg.norm(red[9][:2] - red[5][:2])
        red_r_ext = np.linalg.norm(red[10][:2] - red[6][:2])
        blue_l_ext = np.linalg.norm(blue[9][:2] - blue[5][:2])
        blue_r_ext = np.linalg.norm(blue[10][:2] - blue[6][:2])
        red_stance_width = np.linalg.norm(red[15][:2] - red[16][:2])
        blue_stance_width = np.linalg.norm(blue[15][:2] - blue[16][:2])
        red_pelvis_center = (red[11][:2] + red[12][:2]) / 2
        blue_pelvis_center = (blue[11][:2] + blue[12][:2]) / 2
        red_lean_x = red[0][0] - red_pelvis_center[0]
        blue_lean_x = blue[0][0] - blue_pelvis_center[0]
        red_l_angle = get_angle(red[5][:2], red[7][:2], red[9][:2])
        red_r_angle = get_angle(red[6][:2], red[8][:2], red[10][:2])
        blue_l_angle = get_angle(blue[5][:2], blue[7][:2], blue[9][:2])
        blue_r_angle = get_angle(blue[6][:2], blue[8][:2], blue[10][:2])
        red_l_elbow_y_rel = red[7][1] - red[5][1]
        red_r_elbow_y_rel = red[8][1] - red[6][1]
        blue_l_elbow_y_rel = blue[7][1] - blue[5][1]
        blue_r_elbow_y_rel = blue[8][1] - blue[6][1]
        red_l_wrist_y = red[9][1]
        red_r_wrist_y = red[10][1]
        blue_l_wrist_y = blue[9][1]
        blue_r_wrist_y = blue[10][1]
        return pd.Series({
            'dist_heads': dist_heads, 'red_l_ext': red_l_ext, 'red_r_ext': red_r_ext,
            'blue_l_ext': blue_l_ext, 'blue_r_ext': blue_r_ext,
            'red_stance_width': red_stance_width, 'blue_stance_width': blue_stance_width,
            'red_lean_x': red_lean_x, 'blue_lean_x': blue_lean_x,
            'red_l_angle': red_l_angle, 'red_r_angle': red_r_angle,
            'blue_l_angle': blue_l_angle, 'blue_r_angle': blue_r_angle,
            'red_l_elbow_y_rel': red_l_elbow_y_rel, 'red_r_elbow_y_rel': red_r_elbow_y_rel,
            'blue_l_elbow_y_rel': blue_l_elbow_y_rel, 'blue_r_elbow_y_rel': blue_r_elbow_y_rel,
            'red_l_wrist_y': red_l_wrist_y, 'red_r_wrist_y': red_r_wrist_y,
            'blue_l_wrist_y': blue_l_wrist_y, 'blue_r_wrist_y': blue_r_wrist_y
        })
    except Exception:
        return pd.Series([np.nan]*21, index=[
            'dist_heads', 'red_l_ext', 'red_r_ext', 'blue_l_ext', 'blue_r_ext',
            'red_stance_width', 'blue_stance_width', 'red_lean_x', 'blue_lean_x',
            'red_l_angle', 'red_r_angle', 'blue_l_angle', 'blue_r_angle',
            'red_l_elbow_y_rel', 'red_r_elbow_y_rel', 'blue_l_elbow_y_rel', 'blue_r_elbow_y_rel',
            'red_l_wrist_y', 'red_r_wrist_y', 'blue_l_wrist_y', 'blue_r_wrist_y'
        ])

# Если в репо уже есть готовый final_clean_features.csv — пересчитываем
# фичи только если пришли из extracted_features (ВАРИАНТ A)
if 'red_kps' in full_features_df.columns:
    print("⏳ Пересчитываем боксёрские фичи из ключевых точек...")
    advanced_features = full_features_df.apply(calculate_advanced_features, axis=1)
    df_ml = pd.concat([full_features_df[['video_name', 'frame']], advanced_features], axis=1)
    df_ml = df_ml.sort_values(['video_name', 'frame']).reset_index(drop=True)
    # Скорости
    for side, col in [('red', 'red_l_ext'), ('red', 'red_r_ext'),
                      ('blue', 'blue_l_ext'), ('blue', 'blue_r_ext')]:
        spd = col.replace('_ext', '_speed').replace('_l_', '_l_').replace('_r_', '_r_')
        df_ml[spd.replace('ext','speed').replace('_l_speed','_l_speed').replace('_r_speed','_r_speed')] =             df_ml.groupby('video_name')[col].diff()
    df_ml['red_l_speed'] = df_ml.groupby('video_name')['red_l_ext'].diff()
    df_ml['red_r_speed'] = df_ml.groupby('video_name')['red_r_ext'].diff()
    df_ml['blue_l_speed'] = df_ml.groupby('video_name')['blue_l_ext'].diff()
    df_ml['blue_r_speed'] = df_ml.groupby('video_name')['blue_r_ext'].diff()
    df_ml['red_l_wrist_speed_y'] = df_ml.groupby('video_name')['red_l_wrist_y'].diff()
    df_ml['red_r_wrist_speed_y'] = df_ml.groupby('video_name')['red_r_wrist_y'].diff()
    df_ml['blue_l_wrist_speed_y'] = df_ml.groupby('video_name')['blue_l_wrist_y'].diff()
    df_ml['blue_r_wrist_speed_y'] = df_ml.groupby('video_name')['blue_r_wrist_y'].diff()
    df_ml['red_lean_speed'] = df_ml.groupby('video_name')['red_lean_x'].diff()
    df_ml['blue_lean_speed'] = df_ml.groupby('video_name')['blue_lean_x'].diff()
    df_ml = df_ml.fillna(0)
    df_clean = df_ml[df_ml['dist_heads'] > 0].copy()
    df_clean = df_clean.sort_values(['video_name', 'frame'])
    df_clean['red_l_speed'] = df_clean.groupby('video_name')['red_l_ext'].diff()
    df_clean['red_r_speed'] = df_clean.groupby('video_name')['red_r_ext'].diff()
    df_clean['blue_l_speed'] = df_clean.groupby('video_name')['blue_l_ext'].diff()
    df_clean['blue_r_speed'] = df_clean.groupby('video_name')['blue_r_ext'].diff()
    df_clean['dist_speed'] = df_clean.groupby('video_name')['dist_heads'].diff()
    df_clean['red_l_wrist_speed_y'] = df_clean.groupby('video_name')['red_l_wrist_y'].diff()
    df_clean['red_r_wrist_speed_y'] = df_clean.groupby('video_name')['red_r_wrist_y'].diff()
    df_clean['blue_l_wrist_speed_y'] = df_clean.groupby('video_name')['blue_l_wrist_y'].diff()
    df_clean['blue_r_wrist_speed_y'] = df_clean.groupby('video_name')['blue_r_wrist_y'].diff()
    df_clean['red_lean_speed'] = df_clean.groupby('video_name')['red_lean_x'].diff()
    df_clean['blue_lean_speed'] = df_clean.groupby('video_name')['blue_lean_x'].diff()
    df_clean = df_clean.fillna(0)
    df_clean.to_csv('final_clean_features.csv', index=False)
    print(f"✅ Фичи пересчитаны! Боевых кадров: {len(df_clean)}")
else:
    # ВАРИАНТ B: уже загрузили готовый final_clean_features.csv
    df_clean = full_features_df.copy()
    print(f"✅ Используем готовый датасет из репо. Строк: {len(df_clean)}")

print(f"📊 Итого чистых боевых кадров: {len(df_clean)}")
display(df_clean.head(10))


## 🤖 Шаг 4: Обучение ансамбля и генерация сабмита

In [ ]:
import pandas as pd
import numpy as np
import re
from catboost import CatBoostClassifier
import xgboost as xgb
import warnings
warnings.filterwarnings('ignore')

# ──────────────────────────────────────────────────────────────────────────────
# Файлы берутся из репозитория (склонированного в Шаге 0)
# videos.csv, videos_test.csv, punches.csv, final_clean_features.csv
# ──────────────────────────────────────────────────────────────────────────────
print("1. Загрузка данных из репозитория...")
df_features   = pd.read_csv('final_clean_features.csv')
df_train_meta = pd.read_csv('videos.csv')
df_test_meta  = pd.read_csv('videos_test.csv')   # ← было 'videos (1).csv'
df_all_meta   = pd.concat([df_train_meta, df_test_meta], ignore_index=True)
df_punches    = pd.read_csv('punches.csv')

print(f"  train meta: {len(df_train_meta)} строк")
print(f"  test  meta: {len(df_test_meta)} строк")
print(f"  punches:    {len(df_punches)} строк")
print(f"  features:   {len(df_features)} строк")

def normalize_name(s):
    return re.sub(r'[^a-zA-Zа-яА-Я0-9]', '', str(s)).lower()

df_features['norm_name'] = df_features['video_name'].apply(normalize_name)
df_all_meta['norm_name'] = df_all_meta.apply(
    lambda r: normalize_name(r['source_video'])
    if "IMG_" in str(r['source_video'])
    else normalize_name(str(r['fight_folder']) + str(r['source_video'])), axis=1
)

mapping = df_all_meta.drop_duplicates('norm_name').set_index('norm_name')['video_id'].to_dict()
df_features['video_id'] = df_features['norm_name'].map(mapping)
df_features = df_features.dropna(subset=['video_id']).sort_values(['video_id', 'frame'])

base_cols = [c for c in df_features.columns if c not in ['video_name', 'video_id', 'frame', 'norm_name']]
df_features[base_cols] = df_features[base_cols].fillna(0)

print("2. 🌪️ TIME-SERIES ФИЧИ...")
ts_features = df_features.copy()
speed_cols = [c for c in ts_features.columns if 'speed' in c]

for col in speed_cols:
    ts_features[col + '_shift_1']  = ts_features.groupby('video_id')[col].shift(1).fillna(0)
    ts_features[col + '_shift_m1'] = ts_features.groupby('video_id')[col].shift(-1).fillna(0)
    ts_features[col + '_diff']     = ts_features.groupby('video_id')[col].diff().fillna(0)
    ts_features[col + '_max5']  = ts_features.groupby('video_id')[col].rolling(5, center=True, min_periods=1).max().reset_index(0, drop=True)
    ts_features[col + '_mean5'] = ts_features.groupby('video_id')[col].rolling(5, center=True, min_periods=1).mean().reset_index(0, drop=True)
    ts_features[col + '_mean15'] = ts_features.groupby('video_id')[col].rolling(15, center=True, min_periods=1).mean().reset_index(0, drop=True)
    ts_features[col + '_std15']  = ts_features.groupby('video_id')[col].rolling(15, center=True, min_periods=1).std().fillna(0).reset_index(0, drop=True)

feature_cols = [c for c in ts_features.columns if c not in ['video_name', 'video_id', 'frame', 'norm_name']]

train_ids = df_train_meta['video_id'].unique()
test_ids  = df_test_meta['video_id'].unique()

train_ts = ts_features[ts_features['video_id'].isin(train_ids)].copy()
test_ts  = ts_features[ts_features['video_id'].isin(test_ids)].copy()

print("3. 🎯 Разметка таргета...")
punches_only = df_punches[['video_id', 'frame']].copy()
punches_only['is_punch'] = 1

merged_train = pd.merge_asof(
    train_ts.sort_values('frame'),
    punches_only.sort_values('frame'),
    on='frame', by='video_id', direction='nearest', tolerance=3
)
merged_train['is_punch'] = merged_train['is_punch'].fillna(0).astype(int)

print("4. 🤖 ОБУЧЕНИЕ АНСАМБЛЯ (CatBoost + XGBoost 50/50)...")
pos_weight = (len(merged_train) - merged_train['is_punch'].sum()) / max(1, merged_train['is_punch'].sum())

print("   -> Тренируем CatBoost Детектор...")
cb_detector = CatBoostClassifier(
    iterations=800, depth=6, learning_rate=0.05,
    scale_pos_weight=pos_weight, random_seed=42, verbose=0, task_type='GPU'
)
cb_detector.fit(merged_train[feature_cols], merged_train['is_punch'])

print("   -> Тренируем XGBoost Детектор...")
xgb_detector = xgb.XGBClassifier(
    n_estimators=800, max_depth=6, learning_rate=0.05,
    scale_pos_weight=pos_weight, random_state=42,
    tree_method='hist', device='cuda'
)
xgb_detector.fit(merged_train[feature_cols], merged_train['is_punch'])

print("5. 🛡️ АБСОЛЮТНЫЙ ЩИТ (ансамблевый отбор кадров)...")
cb_probs  = cb_detector.predict_proba(test_ts[feature_cols])[:, 1]
xgb_probs = xgb_detector.predict_proba(test_ts[feature_cols])[:, 1]
test_ts['punch_prob'] = (cb_probs + xgb_probs) / 2.0

top_1594 = test_ts.sort_values('punch_prob', ascending=False).head(1594).copy()
top_1594 = top_1594.sort_values('punch_prob', ascending=False).reset_index(drop=True)
top_1594['clear'] = 'false'
top_1594.loc[:749, 'clear'] = 'true'
top_1594 = top_1594.sort_values(['video_id', 'frame']).reset_index(drop=True)
print("🎯 Выставлено clear='true' для 750 ансамблевых кадров!")

print("6. 🌳 ОБУЧЕНИЕ КЛАССИФИКАТОРОВ АТРИБУТОВ (CatBoost)...")
train_exact = pd.merge_asof(
    train_ts.sort_values('frame'),
    df_punches[['video_id', 'frame', 'fighter', 'punch_type', 'hand', 'target', 'effectiveness']].sort_values('frame'),
    on='frame', by='video_id', direction='nearest', tolerance=5
).dropna(subset=['punch_type'])

targets = ['fighter', 'punch_type', 'hand', 'target', 'effectiveness']

for target in targets:
    print(f"   -> Учим классификатор для: {target}...")
    model_cb = CatBoostClassifier(
        iterations=900, depth=7, learning_rate=0.04, random_seed=42, verbose=0, task_type='GPU'
    )
    model_cb.fit(train_exact[feature_cols], train_exact[target].astype(str))
    top_1594[target] = model_cb.predict(top_1594[feature_cols]).flatten()

print("7. 🏆 Формируем финальный файл...")
final_sub = top_1594.merge(df_test_meta[['video_id', 'agn_index', 'video_key']], on='video_id', how='left')
final_sub['id'] = range(1, 1595)
final_sub = final_sub[['id', 'video_id', 'agn_index', 'video_key', 'frame',
                        'fighter', 'punch_type', 'hand', 'target', 'effectiveness', 'clear']]

final_sub.to_csv('submission_TITAN_ENSEMBLE.csv', index=False)
print("✅ ГОТОВО! Файл: submission_TITAN_ENSEMBLE.csv")
display(final_sub.head(10))
